# Test several `python`, `dask`, `rs-dpr-service` versions

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1044

Now we have 4 different Jupyter venvs/kernels: 

  1. `Python 3`: the main environment (as before) with python, dask, prefect, rs-client-libraries, ...
  1. `py3.11.7-2024.5.2`: a venv with only `python==3.11.7` and `dask==2024.5.2`, used to init the dask clusters for `l0`, `s3olci`, `mockup`
  1. `py3.11.7-2026.1.2`: a venv with only `python==3.11.7` and `dask==2026.1.2`, used to init the dask clusters for `cpm2`
  1. `py3.13.12-2026.3.0`: a venv with only `python==3.13.12` and `dask==2026.3.0`, used to init the dask clusters for `staging`, `s1ard`, `cpm3`

We also have 3 different `rs-dpr-service` instances, one for each sub-venv, with the same versions of python and dask:
  1. `rs-dpr-service_py3.11.7-2024.5.2`
  1. `rs-dpr-service_py3.11.7-2026.1.2`
  1. `rs-dpr-service_py3.13.12-2026.3.0`

Notes: 

  * These venvs are defined in https://github.com/RS-PYTHON/rs-workflow-env/blob/develop/docker/scripts/dask-cluster-versions.yml
  * We have specific notebooks to init each dask cluster using the right venv in the folder: `notebooks/init-dask-clusters/`
  * The configuration to call the right `rs-dpr-service` instance for each processor is made in :

    * https://github.com/RS-PYTHON/rs-server-deployment/tree/develop/apps (cluster mode)
    * https://github.com/RS-PYTHON/rs-demo/blob/develop/local-mode/nginx.conf#L99 (local mode)

In this demo we will:

  1. Test the init of sub-venv dask clusters from the main Jupyter env.
  1. Call rs-dpr-service for each cluster and check that the right instance was used.
  1. Call the init of the staging cluster and check that it retrieves the instance that is automatically init by the infra.

## DPR dask clusters

In [ ]:
# Init dask clusters for each sub-venv. Check the versions of python and dask in the logs.
# Use minimal ram and cpu.
from resources.dask_clusters.dask_main_env import *
cluster_info_l0 = await init_dask_cluster_l0(
    scale=1, worker_cores=1, worker_memory=2.0, scheduler_memory_limit=2, big_resources=False)

# Print cluster information
print(cluster_info_l0)

In [ ]:
cluster_info_cpm2 = await init_dask_cluster_cpm2(
    scale=1, worker_cores=1, worker_memory=2.0, scheduler_memory_limit=2, big_resources=False)
print(cluster_info_cpm2)

In [ ]:
cluster_info_s1ard = await init_dask_cluster_s1ard(
    scale=1, worker_cores=1, worker_memory=2.0, scheduler_memory_limit=2, big_resources=False)
print(cluster_info_s1ard)

Print the `rs-dpr-service` logs with: 

  * `kubectl -n processing logs -f --selector app.kubernetes.io/name=rs-dpr-service --all-pods=true --tail=-1` (cluster mode)
  * `docker compose logs -f $(docker compose ps | awk '{print $1}' | grep rs-dpr-service)` (local mode)

Then call rs-dpr-service to run a dummy s1l0 task. Check in the logs that:

  * The `rs-dpr-service_py3.11.7-2024.5.2` instance is called.
  * It can connect to your cluster with e.g.:

    ```
    Cluster list for gateway 'http://traefik-dask-gateway.dask-gateway.svc.cluster.local': [ClusterReport<name=19953d5ab5764c52bdbb830c069d5a2f, status=RUNNING>]
    Successfully connected to the 'dask-l0.<user>.latest' dask cluster
    Cluster dashboard: http://traefik-dask-gateway.dask-gateway.svc.cluster.local/clusters/dask-gateway.19953d5ab5764c52bdbb830c069d5a2f/status
    ```

  * It then fails because we gave dummy processing parameters, but this is normal.


In [ ]:
from resources.utils import *
init_demo()
from resources.utils import * # reload the global vars again

dpr_client.run_process("s1_l0", cluster_info_l0, "", "", "")

In [ ]:
# Do the same for cpm2 and rs-dpr-service_py3.11.7-2026.1.2
# This one fails before connecting to the cluster, so just check that rs-dpr-service_py3.11.7-2026.1.2 is used.
dpr_client.run_process("conv_safe_zarr", cluster_info_cpm2, "", "", "")

In [ ]:
# Do the same for s1ard and rs-dpr-service_py3.13.12-2026.3.0
dpr_client.run_process("s1_ard", cluster_info_s1ard, "", "", "")

### Show what happens if we init a cluster with the wrong dask version

(open a notebook to init a cluster, change the kernel to use a different dask version, then init the cluster).

---

### Shutdown the clusters by opening their notebooks and calling the last cell.

## Staging dask clusters

Call the init of the staging cluster and check that it retrieves the instance that is automatically init by the infra:

```
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
image = a798e5525f79429d9bbf059481075bd4
Get existing dask cluster: 'a798e5525f79429d9bbf059481075bd4'
Dask dashboard for 'dask-staging': http://localhost/dask/staging/clusters/a798e5525f79429d9bbf059481075bd4/status
Dask workers for 'dask-staging' are up: 2/2
```

In [ ]:
await init_dask_cluster_staging()